In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import joblib

# 1. TextCNN Architecture Definition
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes=[2, 3, 4], num_filters=100):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embed_dim)) for fs in filter_sizes
        ])
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        x = x.unsqueeze(1)     # (batch_size, 1, seq_len, embed_dim)
        
        pooled_outputs = []
        for conv in self.convs:
            c = torch.relu(conv(x)).squeeze(3)
            p = torch.max(c, dim=2)[0]
            pooled_outputs.append(p)
            
        x = torch.cat(pooled_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x)
        return logits

# 2. Model Saving function to integration folder 'models/'
def save_cnn_model(model, tokenizer, filepath="../models/text_cnn_model.pkl"):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    joblib.dump({"model_state": model.state_dict(), "tokenizer": tokenizer}, filepath)
    print(f"CNN Model successfully saved to {filepath}")